# Prepare CHP data

This prepares yearly CHP generation in MWh per country and technology based on the Eurostat CHP questionaire from https://ec.europa.eu/eurostat/documents/38154/4956229/CHPdata2005-2017.xlsx/

- Since coal is not differentiated, we create shares based on yearly entsoe generation data from parse_generation_entsoe_sftp!

- Furthermore, we prepare yearly CHP profile from original data (lion's?)

Careful: '../parsed_data/generation_'+year+'_annual_entsoe.csv' is needed for 2015-2017!

In [1]:
import pandas as pd
import numpy as np
import wget
import os

In [2]:
dir_in = "../source_data/chp/"
dir_out = "../parsed_data/"
years = ['2015','2016','2017'] #careful if extending list of years as SK 2014 is missing
fn = "CHPdata2005-2017.xlsx"
fn_heat_dem = "../source_data/chp/chp_data.xlsx"

In [3]:
#only if data has to be downloaded again:
#url = "https://ec.europa.eu/eurostat/documents/38154/4956229/CHPdata2005-2017.xlsx/871cc151-5733-423f-ae38-de9b733aa81e"
#os.remove(dir_in+fn)
#wget.download(url,dir_in+fn)

! if data is downloaded again, sheet 2017 needs "2017" in cell A3, otherwise, 2017 values are missing

! should also copy all PJ data to row 37 to be consistent

In [4]:
map_country_ISO = {
	"Austria" : "AT",
	"Belgium" : "BE",
	"Belgium1" : "BE",
	"Bulgaria" : "BG",
	"Croatia" : "HR",
	"Cyprus" : "CY",
	"Czech" : "CZ",
	"Czech Republic" : "CZ",
	"Czechia" : "CZ",
	"Denmark" : "DK",
	"Estonia" : "EE",
	"Estonia2" : "EE",
	"Finland" : "FI",
	"France" : "FR",
	"Germany" : "DE",
	"Germany1" : "DE",
	"Germany1, 2" : "DE",
	"Greece" : "GR",
	"Greece2" : "GR",
	"Hungary" : "HU",
	"Hungary1" : "HU",
	"Hungary3" : "HU",
	"Ireland" : "IE",
	"Ireland1" : "IE",
	"Italy" : "IT",
	"Latvia" : "LV",
	"Lithuania" : "LT",
	"Luxembourg" : "LU",
	"Malta" : "MT",
	"Netherlands" : "NL",
	"Norway" : "NO",
	"Norway2" : "NO",
	"Poland" : "PL",
	"Portugal" : "PT",
	"Portugal1" : "PT",
	"Romania" : "RO",
	"Slovakia" : "SK",
	"Slovakia3" : "SK",
	"Slovenia" : "SI",
	"Slovenia1" : "SI",
	"Spain" : "ES",
	"Sweden" : "SE",
	"Sweden2" : "SE",
	"United Kingdom" : "GB",
    "Estonia*" : "EE"
}

In [5]:
#lame fix for column naming:
year_to_country = {
    2015 : "country",
    2016 : "country",
    2017 : "country",
}

In [6]:
df=pd.DataFrame()
for year in years:
    df_temp = pd.read_excel(dir_in + fn, sheet_name = year,
                            header=2, usecols = "A:E", nrows=30,na_values=":") 
    df_temp = df_temp.rename(columns = year_to_country)
    df_temp['year'] = year
    df = df.append(df_temp, ignore_index=True)

Rename countries to ISO and remove non listed countries or regions

In [7]:
df.index = df['country']
df = df.rename(index=map_country_ISO)
df = df[df['country'].isin(map_country_ISO)].drop(columns='country').reset_index()

Convert CHP generation to MWh and drop non needed columns

In [8]:
df['CHP_MWh'] = df['CHP electricity generation, TWh']*1000*1000
df_chp_gen = df[['year','country','CHP_MWh']].copy().set_index(['year','country'])

now we copy 2016 data for NO as this is missing in 2017

In [9]:
df_chp_gen.loc[('2017','NO'),:] = df_chp_gen.loc[('2016','NO'),:]

In [10]:
df_chp_gen.head(1)

,,CHP_MWh
year,country,
2015,BE,12479000.0


Now we also load the per technology share from the same file

In [11]:
df=pd.DataFrame()
for year in years:
    df_temp = pd.read_excel(dir_in + fn, sheet_name = year,
                            header=0, usecols = "A:H", skiprows=36, na_values=":") 
    df_temp = df_temp.rename(columns = year_to_country)
    df_temp['year'] = year
    df = df.append(df_temp, ignore_index=True)

and rename country to code again

In [12]:
df.index = df['country']
df = df.rename(index=map_country_ISO)
df = df[df['country'].isin(map_country_ISO)]

rename columns and reduce df to needed data

In [13]:
df['Oil'] = df['Oil and oil products']
df['Gas'] = df['Natural gas']
df['Lignite'] = df['Solid fossil fuels and peat'] 
df['HardCoal'] = df['Solid fossil fuels and peat']
df['Other'] = df['Other fuels']
df['Biomass'] = df['Renewables']
df_chp_share = df[['year','Oil','Gas','Lignite','HardCoal','Other','Biomass']].reset_index().set_index(['year','country'])
#norway is missing for 2017 so we copy 2016 values before stacking
df_chp_share.loc[('2017','NO'),:] = df_chp_share.loc[('2016','NO'),:]
df_chp_share = pd.DataFrame(df_chp_share.stack()).reset_index()
df_chp_share = df_chp_share.rename(columns={'level_2':'tech',0:'share'})
df_chp_share.head(1)

,year,country,tech,share
0,2015,BE,Oil,0.015


Important: this is on a fuel level and not on an output level, so we use average efficiencies per technology

In [23]:
efficiency = {
    'Nuclear':0.43,
    'Lignite':0.43,
    'Gas':0.60,
    'Oil':0.40,
    'Biomass':0.45,
    'HardCoal':0.46,
    'Other':0.45}
efficiency

{'Nuclear': 0.43,
 'Lignite': 0.43,
 'Gas': 0.6,
 'Oil': 0.4,
 'Biomass': 0.45,
 'HardCoal': 0.46,
 'Other': 0.45}

In [24]:
df_chp_share['efficiency'] = df_chp_share['tech'].map(efficiency)
df_chp_share.head()

,year,country,tech,share,efficiency
0,2015,BE,Oil,0.015,0.40
1,2015,BE,Gas,0.559,0.60
2,2015,BE,Lignite,0.012,0.43
3,2015,BE,HardCoal,0.012,0.46
4,2015,BE,Other,0.241,0.45


now merge the two dfs

In [14]:
df_chp_tech = df_chp_share.merge(df_chp_gen,on=['year','country'])
df_chp_tech.head()

,year,country,tech,share,CHP_MWh
0,2015,BE,Oil,0.015,12479000.0
1,2015,BE,Gas,0.559,12479000.0
2,2015,BE,Lignite,0.012,12479000.0
3,2015,BE,HardCoal,0.012,12479000.0
4,2015,BE,Other,0.241,12479000.0


eventually, for more accurate data, we calculate shares for hard coal and lignite based on their share in yearly generation

In [15]:
df_gen = pd.DataFrame()
for year in years:
    df_gen_temp = pd.read_csv('../parsed_data/generation_'+year+'_annual_entsoe.csv')
    df_gen_temp['year'] = year
    df_gen = df_gen.append(df_gen_temp)
df_gen = df_gen.groupby(['year','country','tech']).sum()
df_gen.head()

output    demand  net_generation
year country tech                                        
2015 AT      Biomass   2.411676  0.000000        2.411676
             Gas       7.641010  0.000000        7.641010
             HardCoal  1.675485  0.000000        1.675485
             Oil       0.000000  0.000000        0.000000
             Other     5.385637  3.391247        1.994389

In [16]:
df_gen_coal = df_gen.reset_index()
df_gen_coal = df_gen_coal[df_gen_coal.tech.isin(['HardCoal','Lignite'])]
df_gen_coal = df_gen_coal[['year','country','tech','net_generation']]
df_gen_coal = df_gen_coal.pivot_table(index=['year','country'], columns='tech',values='net_generation')
df_gen_coal = df_gen_coal.fillna(0)
df_gen_coal['HardCoal_share'] = df_gen_coal['HardCoal'] / (df_gen_coal['HardCoal'] + df_gen_coal['Lignite'])
df_gen_coal['Lignite_share'] = df_gen_coal['Lignite'] / (df_gen_coal['HardCoal'] + df_gen_coal['Lignite'])
df_gen_coal = df_gen_coal.reset_index().set_index(['year','country'])
df_gen_coal = df_gen_coal[['HardCoal_share','Lignite_share']].rename(columns={'HardCoal_share':'HardCoal','Lignite_share':'Lignite'})
df_gen_coal = pd.DataFrame(df_gen_coal.stack()).rename(columns={0:'coal_share'})
df_gen_coal.head()

coal_share
year country tech                
2015 AT      HardCoal         1.0
             Lignite          0.0
     BE      HardCoal         1.0
             Lignite          0.0
     BG      HardCoal         0.0

now merge this into chp df, calculate the yearly CHP generation per technology and clean up

In [17]:
df_chp_gen_tech = df_chp_tech.merge(df_gen_coal.reset_index(),how='left', on=['year','country','tech'])
df_chp_gen_tech.coal_share = df_chp_gen_tech.coal_share.fillna(1)
df_chp_gen_tech['MWh'] = df_chp_gen_tech['share'] * df_chp_gen_tech['CHP_MWh'] * df_chp_gen_tech['coal_share']
df_chp_gen_tech = df_chp_gen_tech[['year','country','tech','MWh']]
df_chp_gen_tech.head()

,year,country,tech,MWh
0,2015,BE,Oil,187185
1,2015,BE,Gas,6.97576e+06
2,2015,BE,Lignite,0
3,2015,BE,HardCoal,149748
4,2015,BE,Other,3.00744e+06


## Now we also create hourly profiles

In [18]:
df_heat_demand_in = pd.read_excel(fn_heat_dem)
df_heat_demand_in.head(1)

,date,heat_demand
0,2014-01-01 00:00:00+00:00,0.86235


In [19]:
df_heat_demand = df_heat_demand_in.drop("date", axis = 1)
df_heat_demand["heat_demand_relative"] = df_heat_demand["heat_demand"]/df_heat_demand["heat_demand"].sum()
df_heat_demand.tail()

,heat_demand,heat_demand_relative
8755,0.896596,0.000153
8756,1.000000,0.000171
8757,1.000000,0.000171
8758,1.000000,0.000171
8759,1.000000,0.000171


# Export both to csv

In [20]:
df_chp_gen_tech.to_csv(dir_out + "chp_generation.csv", encoding="utf-8", index=False)

In [21]:
df_heat_demand.to_csv(dir_out + "heat_demand.csv", encoding="utf-8", index = False)